# Reduce words to their base form so that "feminists", "feminist", "feminism" all count as the same word.
Chai states that lemmatization is better than stemming because it produces real words. However, I will need POS tagging first, because without this, the lemmatizer makes mistakes. 

Example: "better" → "good", "leaves" → "leaf" (noun) or "leave" (verb)

#### 1. Imports
Using this: https://www.geeksforgeeks.org/python/python-lemmatization-with-nltk/

In [23]:
import pandas as pd
import ast
import nltk

nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")

from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/sophiehamann/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/sophiehamann/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/sophiehamann/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


#### 2. Loading the Dataset

In [24]:
df_h1 = pd.read_csv("/Users/sophiehamann/master-thesis-code/data/processed/08_h1_stopwords.csv")
df_h1["tokens"] = df_h1["tokens"].apply(ast.literal_eval)

#### 3. Helper function to convert POS tags
Using this: https://www.geeksforgeeks.org/nlp/nlp-part-of-speech-default-tagging/

In [25]:
# NLTK uses its own POS tag format, but WordNet uses a different one
# this function converts between the two

def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # default to noun

#### 4. Writing the lemmatization function

In [26]:
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    
    # first get POS tag for each token
    pos_tags = nltk.pos_tag(tokens)
    
    # then lemmatize each token using its POS tag
    lemmatized = []
    for token, tag in pos_tags:
        wordnet_tag = get_wordnet_pos(tag)
        lemma = lemmatizer.lemmatize(token, wordnet_tag)
        lemmatized.append(lemma)
    
    return lemmatized

#### 5. Checking before and after in an example row

In [27]:
# check what lemmatization does to your actual text before applying to everything
sample = df_h1["tokens"].iloc[3]
print("BEFORE:", sample)
print("AFTER: ", lemmatize_tokens(sample))

BEFORE: ['feminist', 'art', 'politics', 'published', 'winter', 'spring', 'summer']
AFTER:  ['feminist', 'art', 'politics', 'publish', 'winter', 'spring', 'summer']


#### 6. Applying the function

In [28]:
df_h1["tokens"] = df_h1["tokens"].apply(lemmatize_tokens)

#### 7. Checking

In [29]:
print(df_h1["tokens"].iloc[0])
print(f"H1 total tokens: {df_h1['tokens'].apply(len).sum()}")

['devote', 'examination', 'art', 'politics']
H1 total tokens: 80180


##### The following check has been written by Claude AI Chatbot. 
The prompt can be found here: 

In [30]:
# ── Verification: did lemmatization work correctly? ──────────────────────────

import random
from collections import Counter

lemmatizer_check = WordNetLemmatizer()

# 1. KNOWN WORD PAIRS: words that should have been reduced to a common base
#    Format: (inflected_form, expected_lemma, wordnet_pos)
known_pairs = [
    ("feminists",  "feminist", wordnet.NOUN),
    ("feminism",   "feminist", wordnet.NOUN),   # NOTE: WordNet won't merge these — good to know
    ("heresies",   "heresy",   wordnet.NOUN),
    ("published",  "publish",  wordnet.VERB),
    ("running",    "run",      wordnet.VERB),
    ("better",     "good",     wordnet.ADJ),
    ("leaves",     "leaf",     wordnet.NOUN),
    ("leaves",     "leave",    wordnet.VERB),
]

print("=== 1. Spot-check known word pairs ===")
for word, expected, pos in known_pairs:
    result = lemmatizer_check.lemmatize(word, pos)
    status = "✓" if result == expected else "✗"
    print(f"  {status}  '{word}' ({pos}) → '{result}'  (expected: '{expected}')")

# 2. NO INFLECTED FORMS REMAIN: check that common suffixes were reduced
#    (plurals -s/-es, past tense -ed, gerunds -ing)
all_tokens = [t for tokens in df_h1["tokens"] for t in tokens]
token_counts = Counter(all_tokens)

suspicious = [
    word for word, count in token_counts.items()
    if (word.endswith("ing") or word.endswith("ings") or
        word.endswith("ies") or word.endswith("ations"))
    and len(word) > 6
    and count > 5  # only flag frequent ones
]
print(f"\n=== 2. Potentially un-lemmatized tokens (freq > 5) ===")
if suspicious:
    for w in sorted(suspicious, key=lambda x: -token_counts[x])[:20]:
        print(f"  '{w}': {token_counts[w]}x")
else:
    print("  None found — looks clean!")

# 3. VARIANTS COLLAPSED: do word families now share a single form?
word_families = [
    ["feminist", "feminists"],
    ["heresy", "heresies"],
    ["publish", "published", "publishing"],
    ["art", "arts"],
    ["politic", "politics"],   # 'politics' is tricky — often stays as-is
]

print("\n=== 3. Are word-family variants collapsed into one form? ===")
for family in word_families:
    present = {w: token_counts.get(w, 0) for w in family}
    print(f"  {present}")

# 4. RANDOM SAMPLE: eyeball 5 random rows before vs after
print("\n=== 4. Random sample of lemmatized token lists ===")
for i in random.sample(range(len(df_h1)), min(5, len(df_h1))):
    print(f"  row {i}: {df_h1['tokens'].iloc[i][:10]}")

=== 1. Spot-check known word pairs ===
  ✓  'feminists' (n) → 'feminist'  (expected: 'feminist')
  ✗  'feminism' (n) → 'feminism'  (expected: 'feminist')
  ✓  'heresies' (n) → 'heresy'  (expected: 'heresy')
  ✓  'published' (v) → 'publish'  (expected: 'publish')
  ✓  'running' (v) → 'run'  (expected: 'run')
  ✓  'better' (a) → 'good'  (expected: 'good')
  ✓  'leaves' (n) → 'leaf'  (expected: 'leaf')
  ✓  'leaves' (v) → 'leave'  (expected: 'leave')

=== 2. Potentially un-lemmatized tokens (freq > 5) ===
  'something': 71x
  'morning': 39x
  'meeting': 39x
  'everything': 33x
  'nothing': 29x
  'feeling': 28x
  'anything': 28x
  'building': 27x
  'painting': 26x
  'housing': 18x
  'drawing': 15x
  'reading': 15x
  'upcoming': 15x
  'interesting': 13x
  'planning': 13x
  'evening': 12x
  'writing': 12x
  'understanding': 11x
  'beginning': 10x
  'speaking': 9x

=== 3. Are word-family variants collapsed into one form? ===
  {'feminist': 278, 'feminists': 1}
  {'heresy': 0, 'heresies': 0}
 

#### 8. Saving

In [31]:
df_h1.to_csv("/Users/sophiehamann/master-thesis-code/data/processed/09_h1_lemmatized.csv", index=False)